In [4]:
import pandas as pd
from rdflib import Graph, RDF, URIRef, Namespace
from rdflib.namespace import OWL
import urllib.parse
import os

INPUT_CSV = "data/acquisition/extracted_knowledge.csv"
OUTPUT_TTL = "data/KB_construction/initial_graph.ttl"

def create_initial_graph(csv_file):
    print(f"Creating initial RDF graph from {csv_file}")
    os.makedirs("data/KB_construction", exist_ok=True)
    df = pd.read_csv(csv_file)
    print("Columns found in CSV", df.columns.tolist())
    g = Graph()
    MY_NS = Namespace("http://myproject.org/ontology/")
    g.bind("my", MY_NS)
    g.bind("owl", OWL)
    for index, row in df.iterrows():
        entity_str = str(row['Entity']).strip()
        type_str = str(row['Type']).strip()
        s_uri = URIRef(MY_NS[urllib.parse.quote(entity_str.replace(" ", "_"))])
        o_uri = URIRef(MY_NS[type_str])
        g.add((s_uri, RDF.type, o_uri))
    g.serialize(destination=OUTPUT_TTL, format="turtle")
    print(f"Initial graph saved in {OUTPUT_TTL} with {len(g)} triples")
    return g

initial_graph = create_initial_graph(INPUT_CSV)

Creating initial RDF graph from data/acquisition/extracted_knowledge.csv
Columns found in CSV ['Entity', 'Type', 'Source_URL']
Initial graph saved in data/KB_construction/initial_graph.ttl with 216 triples


In [5]:
import requests
import time
import csv
import pandas as pd
from rdflib import URIRef, Namespace
from rdflib.namespace import OWL
import urllib.parse
import re

MAPPING_CSV = "data/KB_construction/entity_mapping.csv"
ALIGNED_TTL = "data/KB_construction/aligned_graph.ttl"

def clean_entity_name(name):
    name = re.sub(r'\[\d+\]?', '', name)
    name = name.replace("'s", "")
    return name.strip()

def search_wikidata(entity_name):
    url = "https://www.wikidata.org/w/api.php"
    clean_name = clean_entity_name(entity_name)
    if not clean_name:
        return None, 0.0
    params = {
        "action": "wbsearchentities",
        "format": "json",
        "language": "en",
        "search": clean_name,
        "limit": 1
    }
    headers = {
        "User-Agent": "MyStudentLabBot/1.0 (Contact student@university.edu)"
    }
    try:
        response = requests.get(url, params=params, headers=headers)
        if response.status_code == 200:
            data = response.json()
            if 'search' in data and len(data['search']) > 0:
                match = data['search'][0]
                return match['id'], 0.95 
        else:
            print(f"HTTP Error {response.status_code} for {clean_name}")
    except Exception as e:
        print(f"Critical API Error for {clean_name} {e}")
    return None, 0.0

def align_entities(graph):
    print("Searching for alignments on Wikidata")
    MY_NS = Namespace("http://myproject.org/ontology/")
    subjects = set(graph.subjects())
    mappings = []
    for s in subjects:
        if isinstance(s, URIRef) and str(s).startswith(str(MY_NS)):
            entity_name = urllib.parse.unquote(str(s).replace(str(MY_NS), "")).replace("_", " ")
            time.sleep(0.2)
            wd_id, confidence = search_wikidata(entity_name)
            if wd_id:
                wd_uri = URIRef(f"http://www.wikidata.org/entity/{wd_id}")
                graph.add((s, OWL.sameAs, wd_uri))
                mappings.append({"Private Entity": str(s), "External URI": f"wd:{wd_id}", "Confidence": confidence})
                print(f"Alignment found {entity_name} to {wd_id}")
    df_map = pd.DataFrame(mappings)
    df_map.to_csv(MAPPING_CSV, index=False)
    print(f"Mapping table saved in {MAPPING_CSV}")
    graph.serialize(destination=ALIGNED_TTL, format="turtle")
    print(f"Aligned graph saved in {ALIGNED_TTL}")
    return graph

aligned_graph = align_entities(initial_graph)

Searching for alignments on Wikidata
Alignment found Caroline to Q16275172
Alignment found June 2024 to Q61312789
Alignment found Sunset to Q166564
Alignment found the European Union.[1 to Q72396472
Alignment found Melvin Capital to Q85784879
Alignment found June 2019 to Q47087599
Alignment found US to Q30
Alignment found February 24 to Q2355
Alignment found January 31, 2012 to Q17982837
Alignment found 2001 to Q1988
Alignment found U.S. to Q30
Alignment found January 22, 2021 to Q69305926
Alignment found Amazon to Q3884
Alignment found 2009 to Q1996
Alignment found March 2019 to Q31275158
Alignment found Ken Griffin's to Q121337037
Alignment found Chewy to Q28130158
Alignment found daily to Q86
Alignment found Binc to Q95121
Alignment found Tenev to Q56291956
Alignment found April 2019 to Q47087596
Alignment found March 2021 to Q61312973
Alignment found February 26, 2021 to Q69305969
Alignment found YouTube to Q866
Alignment found late January to Q137885731
Alignment found 2008 to Q20

In [6]:
from SPARQLWrapper import SPARQLWrapper, JSON
import time
from rdflib import URIRef
from rdflib.namespace import OWL

FINAL_TTL = "data/KB_construction/expanded_graph.ttl"

def expand_graph(graph):
    print("Starting graph expansion via SPARQL")
    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
    sparql.addCustomHttpHeader("User-Agent", "UniversityLabBot/2.0 (Student Project)") 
    new_triples_count = 0
    
    for s, p, o in graph.triples((None, OWL.sameAs, None)):
        if "wikidata.org/entity/" in str(o):
            wd_id = str(o).split("/")[-1]
            print(f"Deep expanding entity {wd_id}")
            
            query = f"""
            SELECT ?p ?o ?s ?p_in WHERE {{
              {{
                wd:{wd_id} ?p ?o .
                FILTER(isIRI(?o))
              }}
              UNION
              {{
                ?s ?p_in wd:{wd_id} .
                FILTER(isIRI(?s))
              }}
            }}
            LIMIT 2500
            """
            sparql.setQuery(query)
            sparql.setReturnFormat(JSON)
            
            try:
                results = sparql.query().convert()
                for result in results["results"]["bindings"]:
                    if "o" in result and "p" in result:
                        p_uri = URIRef(result["p"]["value"])
                        o_uri = URIRef(result["o"]["value"])
                        graph.add((o, p_uri, o_uri))
                        new_triples_count += 1
                        
                    if "s" in result and "p_in" in result:
                        s_uri = URIRef(result["s"]["value"])
                        p_in_uri = URIRef(result["p_in"]["value"])
                        graph.add((s_uri, p_in_uri, o))
                        new_triples_count += 1
                        
            except Exception as e:
                print(f"SPARQL error for {wd_id} {e}")
                
            time.sleep(1.5)
            
    print("Cleaning up potential duplicates")
    graph.serialize(destination=FINAL_TTL, format="turtle")
    print(f"Expansion finished {new_triples_count} new triples added")
    print(f"Final graph size {len(graph)} triples File {FINAL_TTL}")

expand_graph(aligned_graph)

Starting graph expansion via SPARQL
Deep expanding entity Q16275172
Deep expanding entity Q61312789
Deep expanding entity Q166564
Deep expanding entity Q72396472
Deep expanding entity Q85784879
Deep expanding entity Q47087599
Deep expanding entity Q30
Deep expanding entity Q30
Deep expanding entity Q30
Deep expanding entity Q2355
Deep expanding entity Q17982837
Deep expanding entity Q1988
Deep expanding entity Q69305926
Deep expanding entity Q3884
Deep expanding entity Q1996
Deep expanding entity Q31275158
Deep expanding entity Q121337037
Deep expanding entity Q28130158
Deep expanding entity Q28130158
Deep expanding entity Q86
Deep expanding entity Q95121
Deep expanding entity Q56291956
Deep expanding entity Q47087596
Deep expanding entity Q61312973
Deep expanding entity Q69305969
Deep expanding entity Q866
Deep expanding entity Q137885731
Deep expanding entity Q2004
Deep expanding entity Q2002
Deep expanding entity Q77174913
Deep expanding entity Q77174913
Deep expanding entity Q18013